# Woche 07: Boyer-Moore und Preprocessing

**Block B · Woche 07 · Selbststudium · Starter (Skeleton)**

Dieses Notebook setzt die Theorie aus dem Skriptum *„Boyer-Moore und Preprocessing“* (Woche 07) in Python um.
Es ergänzt die Pflichtübung Ü1 (DNA-Strings) und ist in etwa 100 Minuten lösbar. Alle Aufgaben lassen sich mit
dem Skriptum und den Folien der Woche lösen; Abschnittsnummern im Text verweisen auf das Skriptum.

## Lernziele
- Die **Bad-Character-Regel** anwenden und als Tabelle vorberechnen (Skriptum 3 und 6.1)
- Die **Good-Suffix-Regel** verstehen und selbst programmieren (Skriptum 4)
- Beide Regeln zum **Boyer-Moore-Algorithmus** kombinieren und Alignments sowie Vergleiche zählen (Skriptum 5)
- **Naiv gegen Boyer-Moore** an phiX174 und echten Reads vergleichen (Skriptum 7)
- Erklären, wann sich **Preprocessing** lohnt und was **online** und **offline** bedeutet (Skriptum 6)

## Zu implementierende Funktionen
| Teil | Funktion(en) | Skriptum | Zeit |
|---|---|---|---|
| 1 | `bad_character_shift`, `bad_character_table` | 3, 6.1 | ≈ 20 min |
| 2 | `good_suffix_shift`, `match_shift` | 4 | ≈ 25 min |
| 3 | `preprocess`, `boyer_moore_with_counts` | 5, 6 | ≈ 25 min |
| 4 | `compare_algorithms` | 7 | ≈ 20 min |
| 5 | Fragen beantworten | 6.3, 7, 8 | ≈ 10 min |

## Begriffe im Code
- **shift** (Verschiebung): Um so viele Positionen rückt P nach rechts. `shift = 1` heißt: nächstes Alignment.
- **skip** (übersprungene Alignments): So zählt Langmead auf den Folien. Es gilt immer `skip = shift - 1`.
- **Zeichenvergleich**: jeder Vergleich `t[i + j] != p[j]`, genau wie in Woche 6.

**So arbeitest du:** Setup-Zelle ausführen, dann Teil für Teil die Funktionsrümpfe mit `NotImplementedError` durch
deinen Code ersetzen und die Test-Zelle (`%%ipytest`) ausführen, bis alle Tests grün sind.

## 0. Setup
Führe diese Zelle zuerst aus. Fehlende Pakete werden einmalig installiert. Die Datendateien `phix.fa` und
`ERR266411_1.first1000.fastq` aus Woche 6 werden im aktuellen Ordner, in `daten`, in `../daten` und in `..` gesucht.

In [ ]:
# Benötigte Pakete (werden bei Bedarf einmalig installiert)
import importlib, subprocess, sys
for package in ["ipytest", "pytest"]:
    try:
        importlib.import_module(package)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package])

import os
import random
import time

import pytest
import ipytest
ipytest.autoconfig()

DATA_DIRS = [".", "daten", os.path.join("..", "daten"), ".."]


def data_path(filename: str) -> str:
    """Return the path of a data file, searching the folders in DATA_DIRS."""
    for folder in DATA_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{filename} not found in {DATA_DIRS}. Copy it next to this notebook.")


for f in ["phix.fa", "ERR266411_1.first1000.fastq"]:
    print(f"{f}: {data_path(f)}")

### Werkzeuge aus Woche 6 (fertig, nur ausführen)
Diese Funktionen kennst du aus dem Notebook von Woche 6. Sie sind hier fertig vorgegeben, damit du dich auf
Boyer-Moore konzentrieren kannst. `naive_with_counts` ist die Messlatte für Teil 3 und 4.

In [ ]:
def read_genome(filename: str) -> str:
    """Read a FASTA file and return the sequence without header lines."""
    genome = ""
    with open(filename) as f:
        for line in f:
            if not line.startswith(">"):
                genome += line.rstrip()
    return genome


def read_fastq(filename: str) -> tuple[list[str], list[str]]:
    """Read a FASTQ file and return (sequences, qualities)."""
    sequences, qualities = [], []
    with open(filename) as fh:
        while True:
            fh.readline()                   # name line (@...)
            seq = fh.readline().rstrip()
            fh.readline()                   # plus line
            qual = fh.readline().rstrip()
            if len(seq) == 0:
                break
            sequences.append(seq)
            qualities.append(qual)
    return sequences, qualities


def reverse_complement(s: str) -> str:
    """Reverse complement of a DNA string (A<->T, C<->G, N stays N)."""
    complement = {"A": "T", "C": "G", "G": "C", "T": "A", "N": "N"}
    return "".join(complement[base] for base in reversed(s))


def naive_with_counts(p: str, t: str) -> tuple[list[int], int, int]:
    """Naive exact matching. Returns (occurrences, n_alignments, n_comparisons)."""
    occurrences, n_alignments, n_comparisons = [], 0, 0
    for i in range(len(t) - len(p) + 1):
        n_alignments += 1
        match = True
        for j in range(len(p)):
            n_comparisons += 1
            if t[i + j] != p[j]:
                match = False
                break
        if match:
            occurrences.append(i)
    return occurrences, n_alignments, n_comparisons


genome = read_genome(data_path("phix.fa"))
reads, _ = read_fastq(data_path("ERR266411_1.first1000.fastq"))
print(len(genome), len(reads))                                    # 5386 1000
print(naive_with_counts("word", "There would have been a time for such a word"))   # ([40], 41, 46)

## 1. Die Bad-Character-Regel (Skriptum 3 und 6.1)

Boyer-Moore vergleicht die Zeichen eines Alignments **von rechts nach links**. Beim ersten Mismatch an Position `i`
von P steht im Text das Zeichen `c`. Die Regel (Langmead): Verschiebe P, bis (a) der Mismatch zu einem Match wird
oder (b) P ganz an `c` vorbeigerückt ist.

**Aufgabe 1.1:** Schreibe `bad_character_shift(p, i, c)`. Suche in `p[:i]` (links von der Mismatch-Position) das
**am weitesten rechts** stehende Vorkommen von `c` an Position `j`. Rückgabe: `i - j`. Kommt `c` links von `i` nicht
vor, ist die Verschiebung `i + 1`.

**Aufgabe 1.2:** Schreibe `bad_character_table(p, alphabet)`. Rückgabe ist ein Dictionary: für jedes Zeichen `c` des
Alphabets eine Liste mit einem Eintrag pro Position `i` von P. Der Eintrag ist die Zahl der **übersprungenen
Alignments** (`skip = shift - 1`), so wie in Langmeads Tabelle für P = TCGC. Wo `p[i] == c` gilt, gibt es keinen
Mismatch: Trage dort `None` ein (auf der Folie ein Strich).

*Tipps:* `range(i - 1, -1, -1)` läuft von `i - 1` rückwärts bis 0. Für 1.2 kannst du 1.1 wiederverwenden.

In [ ]:
def bad_character_shift(p: str, i: int, c: str) -> int:
    """Shift given by the bad character rule for a mismatch of p[i] against text character c."""
    # Tipp: rechtestes j < i mit p[j] == c suchen; Rückgabe i - j, sonst i + 1
    raise NotImplementedError("TODO: implement bad_character_shift")


def bad_character_table(p: str, alphabet: str = "ACGT") -> dict[str, list[int | None]]:
    """Skips (shift - 1) for every character and position; None where p[i] == c."""
    # Tipp: zwei Schleifen (Zeichen c, Position i); bad_character_shift wiederverwenden
    raise NotImplementedError("TODO: implement bad_character_table")

In [ ]:
# Langmeads Tabelle für P = TCGC (Skriptum 6.1, Tab. 3)
table = bad_character_table("TCGC")
print("     " + "    ".join("TCGC"))
for c in "ACGT":
    print(c, "  ", "  ".join(" -" if v is None else f"{v:2d}" for v in table[c]))

# Bad-Character-Beispiel von Langmeads Folie (Skriptum 3.2)
print(bad_character_shift("CCTTTTGC", 4, "C") - 1)   # Schritt 1: 2 Alignments übersprungen
print(bad_character_shift("CCTTTTGC", 6, "A") - 1)   # Schritt 2: 6 Alignments übersprungen

In [ ]:
%%ipytest

def test_bad_character_shift_slide_example():
    assert bad_character_shift("CCTTTTGC", 4, "C") == 3    # skip 2
    assert bad_character_shift("CCTTTTGC", 6, "A") == 7    # skip 6, P rückt an A vorbei


def test_bad_character_shift_notebook_201():
    assert bad_character_shift("TCAA", 2, "T") == 2        # Beispiel aus Notebook 2.01
    assert bad_character_shift("TCAA", 0, "G") == 1        # Mismatch ganz links: 1 weiter


def test_bad_character_shift_takes_rightmost_occurrence():
    assert bad_character_shift("ACACG", 4, "A") == 2       # A an Position 2, nicht 0
    assert bad_character_shift("ACACG", 4, "T") == 5


def test_bad_character_table_tcgc():
    assert bad_character_table("TCGC") == {
        "A": [0, 1, 2, 3],
        "C": [0, None, 0, None],
        "G": [0, 1, None, 0],
        "T": [None, 0, 1, 2],
    }


def test_bad_character_table_other_alphabet():
    table = bad_character_table("word", alphabet="wordu")
    assert table["u"] == [0, 1, 2, 3]
    assert table["o"] == [0, None, 0, 1]

## 2. Die Good-Suffix-Regel (Skriptum 4)

Sei `t` der Teil von P, der vor dem Mismatch schon gepasst hat: Bei einem Mismatch an Position `i` ist das das
Suffix `p[i+1:]`. Langmeads Regel: Verschiebe P, bis (a) es **keine Mismatches zwischen P und t** mehr gibt oder
(b) P ganz an `t` vorbeigerückt ist.

**Aufgabe 2.1:** Schreibe `good_suffix_shift(p, i)` für einen Mismatch an Position `i`. Probiere die Verschiebungen
`k = 1, 2, ..., len(p)` der Reihe nach und gib das **kleinste** `k` zurück, für das gilt: Für jede Position `j` von t
(also `i + 1 <= j < len(p)`) mit `j - k >= 0` ist `p[j - k] == p[j]`. Positionen mit `j - k < 0` liegen schon links
von P und zählen nicht. Gibt es kein solches `k < len(p)`, ist das Ergebnis `len(p)`.

> Warum `p[j]`? An den Stellen von t stimmen Text und P überein. Statt mit dem Text kann man also mit P selbst
> vergleichen. Genau deshalb lässt sich die Regel **vorab aus P allein** berechnen (Preprocessing).

**Aufgabe 2.2:** Schreibe `match_shift(p)`: die Verschiebung nach einem **vollständigen Treffer**. Dann ist ganz P
das passende Stück t. Das ist derselbe Fall wie ein Mismatch „links von Position 0“, also `good_suffix_shift(p, -1)`.

*Tipp:* `all(... for j in range(...))` prüft eine Bedingung für alle `j`. Überlege dir den Startwert von `j`:
Es muss sowohl `j > i` (j gehört zu t) als auch `j >= k` (j liegt nach der Verschiebung noch unter P) gelten. Ist `i` die letzte Position, ist t leer
und das Ergebnis 1.

In [ ]:
def good_suffix_shift(p: str, i: int) -> int:
    """Shift given by the (weak) good suffix rule for a mismatch at p[i]; t = p[i+1:]."""
    # Tipp: for k in range(1, len(p)): gilt p[j - k] == p[j] für alle j mit i < j < len(p) und j >= k?
    raise NotImplementedError("TODO: implement good_suffix_shift")


def match_shift(p: str) -> int:
    """Shift after a full match of p (good suffix rule with t = p)."""
    raise NotImplementedError("TODO: implement match_shift")

In [ ]:
# Good-Suffix-Beispiel von Langmeads Folie (Skriptum 4.2): P = CTTACTTAC
p = "CTTACTTAC"
print(good_suffix_shift(p, 5) - 1)    # Schritt 1: t = TAC, 3 Alignments übersprungen
print(good_suffix_shift(p, 1) - 1)    # Schritt 2: t = TACTTAC, 3 Alignments übersprungen
print(match_shift(p))                 # nach einem Treffer: 4 weiter (Skriptum 4.3)
print([good_suffix_shift("GTAGCGGCG", i) for i in range(9)])   # ganze Tabelle für Skriptum 5.1

In [ ]:
%%ipytest

def test_good_suffix_slide_example():
    assert good_suffix_shift("CTTACTTAC", 5) == 4     # skip 3
    assert good_suffix_shift("CTTACTTAC", 1) == 4     # skip 3, Präfix CTTAC passt auf Ende von t


def test_good_suffix_empty_t():
    assert good_suffix_shift("GTAGCGGCG", 8) == 1     # Mismatch ganz rechts: t ist leer


def test_good_suffix_putting_it_together():
    assert good_suffix_shift("GTAGCGGCG", 5) == 3     # Folie: gs 2
    assert good_suffix_shift("GTAGCGGCG", 2) == 8     # Folie: gs 7


def test_good_suffix_notebook_201():
    assert good_suffix_shift("ACTA", 0) == 3


def test_good_suffix_tables_like_langmead():
    # Werte aus Langmeads BoyerMoore-Klasse (BoyerMoore.py im Kursrepository), max(1, ...)
    assert [good_suffix_shift("GTAGCGGCG", i) for i in range(9)] == [8, 8, 8, 8, 8, 3, 3, 2, 1]
    assert [good_suffix_shift("AACAAAC", i) for i in range(7)] == [4, 4, 4, 4, 4, 4, 1]


def test_match_shift():
    assert match_shift("ACAC") == 2                    # Beispiel aus Notebook 2.01
    assert match_shift("CTTACTTAC") == 4
    assert match_shift("ACGT") == 4
    assert match_shift("AAAA") == 1

## 3. Boyer-Moore: beide Regeln zusammen (Skriptum 5 und 6)

**Aufgabe 3.1:** Schreibe `preprocess(p, alphabet)`. Sie berechnet alle Verschiebungen **einmal vorab** und gibt
ein Tupel `(bc, gs, ms)` zurück:
- `bc`: Dictionary `bc[c][i] = bad_character_shift(p, i, c)` für jedes Zeichen `c` des Alphabets und jede Position `i`
  (für `p[i] == c` darfst du einen beliebigen Wert eintragen, z. B. 1; er wird nie gebraucht)
- `gs`: Liste `gs[i] = good_suffix_shift(p, i)`
- `ms`: `match_shift(p)`

**Aufgabe 3.2:** Schreibe `boyer_moore_with_counts(p, t, alphabet)`. Rückgabe `(occurrences, n_alignments,
n_comparisons)` wie bei `naive_with_counts`. Ablauf (Skriptum 5.2):
1. Tabellen mit `preprocess` berechnen, dann `i = 0`.
2. Solange `i <= len(t) - len(p)`: ein Alignment zählen und die Zeichen **von rechts nach links** vergleichen
   (jeden Vergleich zählen).
3. Beim ersten Mismatch an Position `j`: `shift = max(1, bc[t[i + j]][j], gs[j])`.
4. Passen alle Zeichen: `i` merken und `shift = ms`.
5. `i += shift`.

Das `alphabet` muss alle Zeichen enthalten, die im Text vorkommen (bei echten Reads auch `N`).

In [ ]:
def preprocess(p: str, alphabet: str = "ACGT") -> tuple[dict[str, list[int]], list[int], int]:
    """Precompute bad character shifts, good suffix shifts and the shift after a match."""
    # Tipp: Dictionary- und List-Comprehensions mit deinen Funktionen aus Teil 1 und 2
    raise NotImplementedError("TODO: implement preprocess")


def boyer_moore_with_counts(p: str, t: str, alphabet: str = "ACGT") -> tuple[list[int], int, int]:
    """Boyer-Moore exact matching. Returns (occurrences, n_alignments, n_comparisons)."""
    # Gerüst wie naive_with_counts, aber: while-Schleife, Vergleiche von rechts, Verschiebung aus den Tabellen
    raise NotImplementedError("TODO: implement boyer_moore_with_counts")


def boyer_moore(p: str, t: str, alphabet: str = "ACGT") -> list[int]:
    """All offsets where p occurs in t (Boyer-Moore)."""
    return boyer_moore_with_counts(p, t, alphabet)[0]

Die folgende Funktion ist **vorgegeben**. Sie verwendet dein `preprocess` und druckt jeden Schritt wie auf
Langmeads Folie „Putting it together“: Offset, Mismatch-Position und die Skips beider Regeln (bc, gs).

In [ ]:
def print_trace(p: str, t: str, alphabet: str = "ACGT") -> None:
    """Print every alignment Boyer-Moore tries, with the skips of both rules."""
    bc, gs, ms = preprocess(p, alphabet)
    print("T:", t)
    i, step = 0, 1
    while i <= len(t) - len(p):
        j = len(p) - 1
        while j >= 0 and t[i + j] == p[j]:
            j -= 1
        if j < 0:
            info, shift = "match", ms
        else:
            skip_bc, skip_gs = bc[t[i + j]][j] - 1, gs[j] - 1
            shift = max(1, skip_bc + 1, skip_gs + 1)
            info = f"mismatch at j={j}  bc: {skip_bc}, gs: {skip_gs}"
        print(f"P: {' ' * i}{p}   step {step}, offset {i}: {info}")
        i += shift
        step += 1


print_trace("GTAGCGGCG", "GTTATAGCTGATCGCGGCGTAGCGGCGAA")   # Skriptum 5.1
print(boyer_moore_with_counts("GTAGCGGCG", "GTTATAGCTGATCGCGGCGTAGCGGCGAA"))
print(naive_with_counts("GTAGCGGCG", "GTTATAGCTGATCGCGGCGTAGCGGCGAA"))

In [ ]:
%%ipytest

WORD_TEXT = "There would have been a time for such a word"
TOGETHER_T = "GTTATAGCTGATCGCGGCGTAGCGGCGAA"


def test_preprocess_shapes():
    bc, gs, ms = preprocess("TCGC")
    assert set(bc) == set("ACGT")
    assert bc["A"] == [1, 2, 3, 4]
    assert bc["T"][1:] == [1, 2, 3]
    assert gs[-1] == 1
    assert ms == 4


def test_boyer_moore_notebook_201():
    assert boyer_moore("CT", "GCTAGCTCTACGAGTCTA") == [1, 5, 7, 15]


def test_boyer_moore_putting_it_together():
    assert boyer_moore_with_counts("GTAGCGGCG", TOGETHER_T) == ([18], 4, 21)


def test_boyer_moore_word_example():
    assert boyer_moore_with_counts("word", WORD_TEXT, alphabet=set(WORD_TEXT)) == ([40], 12, 15)


def test_boyer_moore_overlapping_and_edge_cases():
    assert boyer_moore("AA", "AAAA") == [0, 1, 2]
    assert boyer_moore("ACGT", "ACGT") == [0]
    assert boyer_moore("ACGTA", "ACGT") == []
    assert boyer_moore("CTTACTTAC", "CGTGCCTACTTACTTACTTACTTACGCGAA") == [8, 12, 16]


def test_boyer_moore_equals_naive_on_random_dna():
    rng = random.Random(42)
    for _ in range(200):
        t = "".join(rng.choice("ACGT") for _ in range(300))
        p = "".join(rng.choice("AC") for _ in range(rng.randint(1, 6)))
        assert boyer_moore(p, t) == naive_with_counts(p, t)[0]

## 4. Naiv gegen Boyer-Moore (Skriptum 7)

**Aufgabe 4.1:** Schreibe `compare_algorithms(patterns, t, alphabet)`. Suche jedes Muster mit beiden Algorithmen im
Text `t`, prüfe mit `assert`, dass beide dieselben Offsets liefern, und summiere die Zählwerte. Rückgabe ist ein
Dictionary mit den Schlüsseln

`"hits"` (Zahl der Muster mit mindestens einem Treffer), `"naive_alignments"`, `"naive_comparisons"`,
`"bm_alignments"`, `"bm_comparisons"`.

Danach laufen zwei Experimente: Muster verschiedener Länge aus dem phiX-Genom und die 1000 echten Reads aus Woche 6.

In [ ]:
def compare_algorithms(patterns: list[str], t: str, alphabet: str = "ACGT") -> dict[str, int]:
    """Search all patterns with naive matching and Boyer-Moore and sum up the counts."""
    # Tipp: result = {"hits": 0, "naive_alignments": 0, ...}; dann eine Schleife über patterns
    raise NotImplementedError("TODO: implement compare_algorithms")

In [ ]:
# Experiment 1: je 100 zufällige Muster pro Länge aus dem phiX-Genom (Skriptum 7.1, Tab. 5)
rng = random.Random(7)
print(f"{'length':>6} {'naive al.':>10} {'BM al.':>8} {'naive cmp.':>11} {'BM cmp.':>8} {'factor':>7}")
for length in [5, 10, 20, 50, 100]:
    patterns = [genome[s:s + length] for s in (rng.randint(0, len(genome) - length) for _ in range(100))]
    r = compare_algorithms(patterns, genome)
    print(f"{length:>6} {r['naive_alignments'] / 100:>10.0f} {r['bm_alignments'] / 100:>8.0f} "
          f"{r['naive_comparisons'] / 100:>11.0f} {r['bm_comparisons'] / 100:>8.0f} "
          f"{r['naive_comparisons'] / r['bm_comparisons']:>7.1f}")

In [ ]:
# Experiment 2: 1000 echte Reads (100 nt) gegen phiX, nur der gespeicherte Strang (Skriptum 7.2)
result_reads = compare_algorithms(reads, genome, alphabet="ACGTN")
print(result_reads)

# Laufzeit: Vorverarbeitung und Suche getrennt messen (Werte hängen vom Rechner ab)
start = time.perf_counter()
tables = [preprocess(r, "ACGTN") for r in reads]
t_pre = time.perf_counter() - start

start = time.perf_counter()
for r in reads:
    naive_with_counts(r, genome)
t_naive = time.perf_counter() - start

start = time.perf_counter()
for r in reads:
    boyer_moore_with_counts(r, genome, "ACGTN")
t_bm = time.perf_counter() - start
print(f"naive: {t_naive:.2f} s | Boyer-Moore incl. preprocessing: {t_bm:.2f} s | preprocessing alone: {t_pre:.2f} s")

In [ ]:
%%ipytest

def test_compare_small_example():
    text = "There would have been a time for such a word"
    r = compare_algorithms(["word", "time", "xyz"], text, alphabet=set(text) | set("xyz"))
    assert r["hits"] == 2
    assert r["naive_alignments"] == 41 + 41 + 42
    assert r["bm_alignments"] < r["naive_alignments"]


def test_compare_word_counts():
    text = "There would have been a time for such a word"
    r = compare_algorithms(["word"], text, alphabet=set(text))
    assert r == {"hits": 1, "naive_alignments": 41, "naive_comparisons": 46,
                 "bm_alignments": 12, "bm_comparisons": 15}


def test_compare_phix_reads():
    g = read_genome(data_path("phix.fa"))
    seqs, _ = read_fastq(data_path("ERR266411_1.first1000.fastq"))
    r = compare_algorithms(seqs, g, alphabet="ACGTN")
    assert r == {"hits": 7, "naive_alignments": 5_287_000, "naive_comparisons": 7_072_319,
                 "bm_alignments": 828_850, "bm_comparisons": 1_174_522}


def test_compare_both_strands_like_week_06():
    g = read_genome(data_path("phix.fa"))
    seqs, _ = read_fastq(data_path("ERR266411_1.first1000.fastq"))
    hits = sum(1 for s in seqs if boyer_moore(s, g, "ACGTN") or boyer_moore(reverse_complement(s), g, "ACGTN"))
    assert hits == 449                                  # gleiches Ergebnis wie naiv in Woche 6

## 5. Nachdenken und notieren

Beantworte die Fragen mit deinen Ergebnissen aus Teil 4 und dem Skriptum. Schreibe die Antworten in die Zelle darunter.

1. Um welchen Faktor spart Boyer-Moore bei den Reads Zeichenvergleiche? Warum wächst der Vorteil mit der Musterlänge?
2. In deiner Zeitmessung ist Boyer-Moore samt Vorverarbeitung vielleicht langsamer als der naive Algorithmus.
   Woran liegt das? In welcher Situation lohnt sich die Vorverarbeitung trotzdem?
3. Ordne zu und begründe: naiver Algorithmus, Boyer-Moore, Suchmaschine, Read-Alignment. Online oder offline?
4. Warum springt die Bad-Character-Regel bei DNA seltener weit als bei englischem Text (Skriptum 7.3)?
5. Was müsste ein Algorithmus zusätzlich können, um auch Reads mit Lesefehlern zu finden (Ausblick Woche 9)?

*Deine Antworten:*

1.
2.
3.
4.
5. 